# lora 训练 whisper-v3

## 缓存与路径

In [1]:
from pathlib import Path
import os
import re

# Hugging Face 缓存放到外接盘，避免占用 C 盘
HF_ROOT = "/sigan/ai-ml/huggingface"
os.environ["HF_HOME"] = HF_ROOT
os.environ["HF_HUB_CACHE"] = f"{HF_ROOT}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{HF_ROOT}/datasets"

# Common Voice 27 中文目录
CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)
CLIPS_DIR = CV_ROOT / "clips"

# CV_ROOT = Path(
#     "/sigan/ai-ml/datasets/common-voice-27/chinese/"
#     "common-voice-scripted-speech-27-0-chines-b098f12c.partial/"
#     "cv-corpus-27.0-2026-09-11/zh-CN"
# )
# CLIPS_DIR = CV_ROOT / "clips"

# 如果解压完成后目录去掉了 .partial，把上面的路径改成实际路径
print(CV_ROOT)
print(CLIPS_DIR)

/home/sigan/datasets/common_voice_27_zh/cv-corpus-27.0-2026-09-11/zh-CN
/home/sigan/datasets/common_voice_27_zh/cv-corpus-27.0-2026-09-11/zh-CN/clips


## 加载 TSV 和音频

In [2]:
from datasets import DatasetDict, Audio, load_dataset

# 读取一个 split：TSV 提供文本和文件名，clips 提供真实音频
def load_split(tsv_name):
    ds = load_dataset(
        "csv",
        data_files=str(CV_ROOT / tsv_name),
        delimiter="\t",
        keep_default_na=False,
    )["train"]

    # TSV 中只有文件名，这里补成能访问的绝对路径
    ds = ds.map(
        lambda batch: {
            "audio": [
                str(CLIPS_DIR / filename)
                for filename in batch["path"]
            ]
        },
        batched=True,
    )

    # 把字符串路径变成可解码的 16kHz 音频对象
    ds = ds.cast_column(
        "audio",
        Audio(sampling_rate=16000),
    )

    # 只保留训练真正需要的 audio 和 sentence
    return ds.select_columns(["audio", "sentence"])


common_voice = DatasetDict({
    "train": load_split("train.tsv"),
    "validation": load_split("dev.tsv"),
    "test": load_split("test.tsv"),
})

print(common_voice)
print(common_voice["train"][0])

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 29814
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 10679
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 10679
    })
})


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


{'audio': {'path': '/home/sigan/datasets/common_voice_27_zh/cv-corpus-27.0-2026-09-11/zh-CN/clips/common_voice_zh-CN_19716600.mp3', 'array': array([ 3.09228199e-11, -2.54658516e-11,  1.18234311e-11, ...,
        3.63416984e-05, -9.09159862e-05,  1.47240971e-05]), 'sampling_rate': 16000}, 'sentence': '一般人应该都不陌生'}


## 加载 Processor 并预处理

In [3]:
from transformers import AutoProcessor

MODEL_ID = "openai/whisper-large-v3"

# large-v3 使用 128 个 Mel 频带，必须使用它自己的 Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor


def prepare_dataset(batch):
    audio = batch["audio"]

    # 音频转 Log-Mel 特征，large-v3 输出约 [128, 3000]
    batch["input_features"] = (
        feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
        )
        .input_features[0]
    )

    # 正确文本转 token ID，作为训练 labels
    batch["labels"] = tokenizer(
        batch["sentence"]
    ).input_ids

    return batch


# 29,814 条全量训练数据都处理；外接盘较慢时把 num_proc 改成 2 或 1
tokenized_dataset = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice["train"].column_names,
    num_proc=4,
    writer_batch_size=100,
)

# 验证集有 10,679 条，每次生成评估太慢，固定抽 1,000 条
eval_dataset = (
    tokenized_dataset["validation"]
    .shuffle(seed=42)
    .select(range(300)) # 改成300试试
)

# 用完整训练集训练，最后用完整测试集评估
train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

print("train：", len(train_dataset))
print("validation subset：", len(eval_dataset))
print("test：", len(test_dataset))

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


train： 29814
validation subset： 300
test： 10679


## Data Collator

这组目标模块会同时覆盖 Encoder、Decoder、交叉注意力和 FFN。没有加入 proj_out，因为它是语言模型输出头，参数量大，通常不值得用 LoRA 训练。

In [4]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import jiwer
import torch

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForSpeechSeq2Seq,
    BitsAndBytesConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


# 把不同长度的文本标签补齐，音频特征也用 Processor 对齐
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [
            {"input_features": x["input_features"]}
            for x in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": x["labels"]}
            for x in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        if (
            labels[:, 0]
            == self.processor.tokenizer.bos_token_id
        ).all().cpu().item():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels": labels,
        }


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

## 定义指标

In [5]:
import re
import unicodedata

import opencc

# 繁体转简体
_t2s = opencc.OpenCC("t2s")

# 统一常见中英文标点
_PUNCT_MAP = str.maketrans({
    ",": "，",
    "?": "？",
    "!": "！",
    ":": "：",
    ";": "；",
    "(": "（",
    ")": "）",
    "[": "【",
    "]": "】",
    "{": "｛",
    "}": "｝",
    "、": "，",
})

# 简繁转换与标点归一化函数
def normalize_chinese(text, remove_punctuation=False):
    """
    Chinese ASR 文本统一：
    1. 繁体 -> 简体
    2. 全角字符标准化
    3. 英文转小写
    4. 删除空格
    5. 统一常见标点
    6. 可选删除全部标点
    """
    if text is None:
        return ""

    # 先统一繁简体
    text = _t2s.convert(text)

    # 将全角字母、数字等统一为半角形式
    text = unicodedata.normalize("NFKC", text)

    # 英文大小写统一
    text = text.lower()

    # 中文 ASR 通常不依赖空格
    text = re.sub(r"\s+", "", text)

    # 英文句点不误伤小数，例如 3.14 保持不变
    text = re.sub(r"(?<!\d)\.(?!\d)", "。", text)

    # 统一中英文标点
    text = text.translate(_PUNCT_MAP)

    # 连续标点压缩成一个
    text = re.sub(r"，+", "，", text)
    text = re.sub(r"。+", "。", text)
    text = re.sub(r"？+", "？", text)
    text = re.sub(r"！+", "！", text)

    # 如果只比较识别内容，可以删除全部标点
    if remove_punctuation:
        text = re.sub(
            r"[，。？！：；、（）【】｛｝,\.\?!:;()\[\]{}]",
            "",
            text,
        )

    return text.strip()

In [6]:
# 中文没有天然空格，CER 比 WER 更适合作为主要指标
def compute_metrics(result):
    pred_ids = result.predictions
    label_ids = result.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids = label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    raw_predictions = tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True,
    )
    
    raw_references = tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True,
    )
    
    predictions = [
        normalize_chinese(
            text,
            remove_punctuation=True,
        )
        for text in raw_predictions
    ]
    
    references = [
        normalize_chinese(
            text,
            remove_punctuation=True,
        )
        for text in raw_references
    ]

    # 过滤掉归一化后为空的参考文本，避免 jiwer 报错
    valid_pairs = [
        (prediction, reference)
        for prediction, reference in zip(
            predictions,
            references,
        )
        if reference.strip()
    ]

    empty_reference_count = (
        len(references) - len(valid_pairs)
    )

    # 整个批次都没有有效参考文本时，不计算 CER
    if not valid_pairs:
        return {
            "cer": 1.0,
            "empty_references": float(empty_reference_count),
        }

    valid_predictions, valid_references = zip(
        *valid_pairs
    )

    return {
        "cer": float(
            jiwer.cer(
                reference=list(valid_references),
                hypothesis=list(valid_predictions),
            )
        ),
        "empty_references": float(empty_reference_count),
    }


# 生成 token 后再交给 compute_metrics
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

## 定义模型训练配置

In [7]:
# 8-bit 加载基座，显存更适合 RTX 4080
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True
    ),
    device_map="auto",
)

# 训练时关闭 KV Cache，并清除推理阶段的固定解码限制
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 训练过程中的生成式评估也固定使用中文转写
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="chinese",
    task="transcribe",
)
# 不加这一项，trainer.predict() 可能使用错误的语言任务标记
model.generation_config.forced_decoder_ids = (
    forced_decoder_ids
)

# 开启梯度检查点，进一步降低训练激活显存
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
# 全量线性模块：注意力 Q/K/V/O 加 FFN fc1/fc2
target_modules = [
    "q_proj",
    "k_proj",
    
    # 之前 v3 full 修改全部注意力、交叉注意力和 FFN；
    # 更新范围太大，导致 decoder 生成重复；
    # q/v 更新范围更小，更适合在基座已经较强时做小幅适配。
    # "v_proj",
    # "out_proj",
    # "fc1",
    # "fc2",
]

# 只训练 LoRA 参数，冻结量化后的 Whisper 基座
lora_config = LoraConfig(
    r=8,
    # lora_alpha=32,
    lora_alpha=8,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
)
# 2026-11-10 第一轮 epoch=1 跑完，先做短训练，adapter 已让 Whisper 的 decoder 退化
# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=8,          # 从 32 降到 8，缩放倍率变成 1
#     target_modules=[
#         "q_proj",
#         "v_proj",
#     ],
#     lora_dropout=0.05,
#     bias="none",
# )

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,932,160 || all params: 1,547,422,720 || trainable%: 0.2541102666503436


## 训练

In [9]:
OUTPUT_DIR = (
    "/sigan/ai-ml/models/whisper-large-v3-zh-safe-evalfix"
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # RTX 4080 16GB：物理 batch 2，累积 8 次，有效 batch=16
    per_device_train_batch_size=2,
    # 有效 batch = 2 × 8 = 16
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,

    # 先完整训练 1 轮
    # num_train_epochs=1,
    # 当前 V3 已预训练功能强大，很容易推理退化，步数小点方便随时保存
    max_steps=1000,
    
    # 低学习率，避免破坏 v3 原有生成能力
    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_torch",

    # RTX 4080 支持 FP16，混合精度训练应该开启
    fp16=True,
    bf16=False,

    # 每 1000 个优化步验证一次；日志不会自动保存模型
    # 每 250 步观察 CER 和保存 checkpoint
    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,
    logging_steps=20,

    # 验证时真正生成文本，而不是只计算 teacher-forcing loss
    predict_with_generate=True,
    generation_max_length=225,

    # 用验证集 CER 选择效果最好的 checkpoint
    metric_for_best_model="cer",
    greater_is_better=False,
    load_best_model_at_end=True,

    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)
# 2026-11-10 第一轮 epoch=1 跑完，训练参数改为 300~500 冒烟测试
# training_args = Seq2SeqTrainingArguments(
#     output_dir="/sigan/ai-ml/models/whisper-large-v3-zh-smoke",

#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     per_device_eval_batch_size=2,

#     learning_rate=1e-5,    # 从 1e-4 降下来
#     max_steps=300,
#     warmup_ratio=0.1,
#     lr_scheduler_type="linear",

#     fp16=True,

#     evaluation_strategy="steps",
#     eval_steps=100,
#     save_strategy="steps",
#     save_steps=100,
#     save_total_limit=3,

#     predict_with_generate=True,
#     generation_max_length=225,

#     metric_for_best_model="cer",
#     greater_is_better=False,
#     load_best_model_at_end=True,

#     logging_steps=20,
#     remove_unused_columns=False,
#     label_names=["labels"],
#     report_to="none",
#     seed=42,
# )


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    tokenizer=processor.tokenizer,
)

In [10]:
# 如果已有 checkpoint，改为 trainer.train(resume_from_checkpoint=True)
trainer.train()
# trainer.train(resume_from_checkpoint=True)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss,Cer,Empty References
250,1.433900,1.426521,0.165639,1.000000
500,1.279800,1.276742,0.163877,1.000000
750,1.113300,1.196337,0.161674,1.000000
1000,1.251900,1.172706,0.162775,1.000000


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=1000, training_loss=1.283232147216797, metrics={'train_runtime': 10974.8785, 'train_samples_per_second': 1.458, 'train_steps_per_second': 0.091, 'total_flos': 5.45049280512e+19, 'train_loss': 1.283232147216797, 'epoch': 0.54})

## 保存

In [11]:
# 保存 LoRA adapter 和 Processor；完整 Whisper 基座不需要重复保存
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("已保存：", OUTPUT_DIR)
# trainer.save_model("/sigan/ai-ml/models/whisper-large-v3-zh-smoke")
# processor.save_pretrained("/sigan/ai-ml/models/whisper-large-v3-zh-smoke")
# print("已保存：", "/sigan/ai-ml/models/whisper-large-v3-zh-smoke")

已保存： /sigan/ai-ml/models/whisper-large-v3-zh-safe-evalfix


## 评估

In [12]:
# 验证集只用于调参，最终结果必须看没有参与训练的测试集
test_result = trainer.predict(
    test_dataset,
    metric_key_prefix="test",
    max_length=225,
    num_beams=1,
)

print(test_result.metrics)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


{'test_loss': 1.0363194942474365, 'test_cer': 0.1329369214736713, 'test_empty_references': 0.0, 'test_runtime': 20528.4959, 'test_samples_per_second': 0.52, 'test_steps_per_second': 0.26}


In [19]:
small_test = test_dataset.select(range(50))

small_result = trainer.predict(
    small_test,
    metric_key_prefix="small_test",
    max_length=225,
    num_beams=1,
)

print(small_result.metrics)

{'small_test_loss': 1.2038023471832275, 'small_test_cer': 1.0, 'small_test_empty_references': 0.0, 'small_test_runtime': 100.0455, 'small_test_samples_per_second': 0.5, 'small_test_steps_per_second': 0.25}


# 测试音频

## 第一轮 epoch 跑完测试结果

In [1]:
import csv
from pathlib import Path

import jiwer
import librosa
import torch

from peft import PeftModel
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    BitsAndBytesConfig,
)

MODEL_ID = "openai/whisper-large-v3"

ADAPTER_DIR = (
    "/sigan/ai-ml/models/"
    "whisper-large-v3-zh-commonvoice27-lora"
)

CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

# 加载 Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

# 加载量化基座
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True
    ),
    device_map="auto",
)

# 把 LoRA adapter 挂回 Whisper
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)
model.eval()

# 固定 Whisper 使用中文转写
model.generation_config.forced_decoder_ids = (
    processor.get_decoder_prompt_ids(
        language="chinese",
        task="transcribe",
    )
)


def transcribe(audio_path):
    # 加载并转为 Whisper 要求的 16kHz
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # 8-bit 模型内部偏置是 FP16
    inputs["input_features"] = (
        inputs["input_features"].half()
    )

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=1,
        )

    return processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )[0]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### 测试10条

In [2]:
with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    rows = list(csv.DictReader(
        handle,
        delimiter="\t",
    ))[:10]

references = []
predictions = []

for index, sample in enumerate(rows, start=1):
    audio_path = (
        CV_ROOT
        / "clips"
        / sample["path"]
    )

    reference = sample["sentence"]
    prediction = transcribe(audio_path)

    references.append(reference)
    predictions.append(prediction)

    print(f"\n样本 {index}")
    print("标准：", reference)
    print("预测：", prediction)
    print(
        "CER：",
        jiwer.cer(
            reference=reference,
            hypothesis=prediction,
        ),
    )

overall_cer = jiwer.cer(
    reference=references,
    hypothesis=predictions,
)

print("\n10 条总体 CER：", overall_cer)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename



样本 1
标准： 生于今宁波余姚市南城学弄至东朝街莫家弄。
预测： 生于今宁波于遥遝遝遝遝遝遝遝遝遝遝遝遝遝遝避避避避避避避避避避避避避避避避避避避避避避避避避避避避避避
CER： 2.3

样本 2
标准： 硝酸钍溶液与草酸溶液反应沉淀出草酸钍。
预测： 硝氧融草草氰氰氰氰氰氰氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢氢�
CER： 2.526315789473684

样本 3
标准： 日本的全称为株式会社平凡社。
预测： 日本的全成为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为为。
CER： 5.714285714285714

样本 4
标准： 亿的单位与上下数可参考下表。
预测： 异的单位与与�。
CER： 0.6428571428571429

样本 5
标准： 带广志的名义绘画成人向漫画。
预测： 戴广带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带带�
CER： 3.4285714285714284

样本 6
标准： 满洲国陆军军官学校位于新京。
预测： 满州国陆军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军军写。
CER： 3.142857142857143

样本 7
标准： 埃皮奈人口变化图示
预测： 埃皮奈人口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口口
CER： 10.0

样本 8
标准： 相传民国十一年桥的部分木结构桥身曾被洪水冲走。
预测： 相传民国十一年桥的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的的
CER： 3.739130434782609

样本 9
标准： 根城日本之古城。
预测： 根成日本之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之之

### 结果判断
正常模型生成的 10 条中，应该能看到：
- 输出有合理结尾，不是不断重复；
- 大部分汉字与参考答案接近；
- 总体 CER 明显低于 1.0；
- 不同的音频会生成不同内容。

说明不是 CER 算错，而是 adapter 已让 Whisper 的 decoder 退化

## 第二轮 300 steps 跑完测试结果

In [12]:
import csv
from pathlib import Path

import jiwer
import librosa
import torch

from peft import PeftModel
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    BitsAndBytesConfig,
)

MODEL_ID = "openai/whisper-large-v3"

ADAPTER_DIR = (
    "/sigan/ai-ml/models/whisper-large-v3-zh-smoke"
)

CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

# 加载 Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

# 加载量化基座
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True
    ),
    device_map="auto",
)

# 把 LoRA adapter 挂回 Whisper
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)
model.eval()

# 固定 Whisper 使用中文转写
model.generation_config.forced_decoder_ids = (
    processor.get_decoder_prompt_ids(
        language="chinese",
        task="transcribe",
    )
)


def transcribe(audio_path):
    # 加载并转为 Whisper 要求的 16kHz
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # 8-bit 模型内部偏置是 FP16
    inputs["input_features"] = (
        inputs["input_features"].half()
    )

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=1,
        )

    return processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )[0]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### 测试10条

In [13]:
with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    rows = list(csv.DictReader(
        handle,
        delimiter="\t",
    ))[:10]

references = []
predictions = []

for index, sample in enumerate(rows, start=1):
    audio_path = (
        CV_ROOT
        / "clips"
        / sample["path"]
    )

    reference = sample["sentence"]
    prediction = transcribe(audio_path)

    references.append(reference)
    predictions.append(prediction)

    print(f"\n样本 {index}")
    print("标准：", reference)
    print("预测：", prediction)
    print(
        "CER：",
        jiwer.cer(
            reference=reference,
            hypothesis=prediction,
        ),
    )

overall_cer = jiwer.cer(
    reference=references,
    hypothesis=predictions,
)

print("\n10 条总体 CER：", overall_cer)


样本 1
标准： 生于今宁波余姚市南城学弄至东朝街莫家弄。
预测： 生于金宁波于遥是南城学弄至东朝皆莫加弄。
CER： 0.3

样本 2
标准： 硝酸钍溶液与草酸溶液反应沉淀出草酸钍。
预测： 硝酸土溶液与草酸溶液反应沉淀出草酸土
CER： 0.15789473684210525

样本 3
标准： 日本的全称为株式会社平凡社。
预测： 日本的全程为都市会社平反社
CER： 0.35714285714285715

样本 4
标准： 亿的单位与上下数可参考下表。
预测： E的单位与上下数和参考下表
CER： 0.21428571428571427

样本 5
标准： 带广志的名义绘画成人向漫画。
预测： 戴廣志的名義繪畫成人像漫畫
CER： 0.5714285714285714

样本 6
标准： 满洲国陆军军官学校位于新京。
预测： 满洲国陆军军官学校位于新京
CER： 0.07142857142857142

样本 7
标准： 埃皮奈人口变化图示
预测： IP内人口变化图示
CER： 0.3333333333333333

样本 8
标准： 相传民国十一年桥的部分木结构桥身曾被洪水冲走。
预测： 相传民国十一年桥的部分木结构桥身曾被洪水冲走
CER： 0.043478260869565216

样本 9
标准： 根城日本之古城。
预测： 根城日本之古城
CER： 0.125

样本 10
标准： 物理学者做实验尚未发现任何违背玻恩定则的量子行为。
预测： 物理学者做实验尚未发现任何违背波恩定则的量子行为
CER： 0.08

10 条总体 CER： 0.20625


### 结果判断

用同样的 10 条测试音频做了对比：
- 原始 Whisper large-v3 CER：0.20625
- 300步 smoke LoRA CER：   0.20625

两边总体完全一样，只是个别样本略有变化：
- smoke 把第 1 条改好了一点；
- 第 3 条反而略差；
- 其余样本基本一致。

判断：Whisper large-v3 本身已经很强，Common Voice 这批普通朗读数据对 LoRA 的提升空间很小。 300 步低学习率 LoRA 基本是在保留原模型能力，没有产生明显增益

结论：做一次 50 条对比

### 50条对比

In [1]:
import csv
import gc
import time
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import torch
from IPython.display import display

from peft import PeftModel
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
)

#### 配置模型、数据和测试数量

In [2]:
# 原始 Whisper 模型
MODEL_ID = "openai/whisper-large-v3"

# 300 步 smoke 训练得到的 LoRA adapter
ADAPTER_DIR = (
    "/sigan/ai-ml/models/"
    "whisper-large-v3-zh-smoke"
)

# Common Voice 27 完整中文数据目录
CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

# 本次比较的数据量
NUM_SAMPLES = 50

# 单条语音最多生成多少 token
# 100 对当前这些短句已经足够
MAX_NEW_TOKENS = 100

# 每次只生成一个结果，速度比 beam search 快
NUM_BEAMS = 1

# 统一使用 16kHz，Whisper 固定要求这个采样率
SAMPLING_RATE = 16000

DEVICE = "cuda"

#### 加载 Processor

In [3]:
# Processor 同时包含：
# feature_extractor：音频 -> Log-Mel 特征
# tokenizer：文本 ID -> 文本
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

# Whisper 推理时明确告诉模型：
# language=中文，task=转写
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="chinese",
    task="transcribe",
)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


#### 读取测试集前 50 条，提前处理音频

In [4]:
with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    test_rows = list(
        csv.DictReader(
            handle,
            delimiter="\t",
        )
    )[:NUM_SAMPLES]

print("测试数量：", len(test_rows))

# ============================================================
# 提前处理音频
# ============================================================

# 先把 50 条音频全部转成 Whisper 输入特征。
# 这样原始模型和 LoRA 模型会使用完全相同的输入，避免数据差异。
references = []
input_features = []

for index, row in enumerate(test_rows, start=1):
    # TSV 的 path 只是 MP3 文件名
    audio_path = CV_ROOT / "clips" / row["path"]

    # 读取音频，并统一重采样为 16kHz
    audio, _ = librosa.load(
        str(audio_path),
        sr=SAMPLING_RATE,
    )

    # feature_extractor 输出 input_features
    # [0] 去掉单样本产生的 batch 维度
    feature = processor(
        audio,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
    )["input_features"][0]

    references.append(row["sentence"])
    input_features.append(feature)

    if index % 10 == 0:
        print(f"已处理音频：{index}/{NUM_SAMPLES}")

print("音频预处理完成")

测试数量： 50


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


已处理音频：10/50
已处理音频：20/50
已处理音频：30/50
已处理音频：40/50
已处理音频：50/50
音频预处理完成


####  加载模型并运行 50 条

In [5]:
def run_test(use_lora):
    """
    use_lora=False：测试原始 Whisper
    use_lora=True：测试 Whisper + LoRA adapter
    """

    print("\n开始加载模型，use_lora =", use_lora)

    # 8-bit 加载可以把显存控制在 16GB 内
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_8bit=True
        ),
        device_map="auto",
    )

    # 推理阶段可开启 KV Cache，提高生成速度
    base_model.config.use_cache = True

    model = base_model

    # 只有 LoRA 测试才挂载 adapter
    if use_lora:
        model = PeftModel.from_pretrained(
            base_model,
            ADAPTER_DIR,
        )

    model.eval()

    # 设置 Whisper 中文转写提示
    model.generation_config.forced_decoder_ids = (
        forced_decoder_ids
    )

    predictions = []
    start_time = time.perf_counter()

    for index, feature in enumerate(
        input_features,
        start=1,
    ):
        # 增加 batch 维度：
        # [128, 3000] -> [1, 128, 3000]
        inputs = {
            "input_features": (
                feature
                .unsqueeze(0)
                .to(DEVICE)
                .half()
            )
        }

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
            )

        # token ID -> 中文文本
        prediction = processor.batch_decode(
            output_ids,
            skip_special_tokens=True,
        )[0]

        predictions.append(prediction)

        if index % 5 == 0:
            print(f"已生成：{index}/{NUM_SAMPLES}")

    elapsed = time.perf_counter() - start_time

    # 清理模型，避免下一轮加载时 16GB 显存不足
    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, elapsed


# 原始 Whisper
base_predictions, base_time = run_test(
    use_lora=False
)

# Whisper + smoke LoRA
lora_predictions, lora_time = run_test(
    use_lora=True
)


开始加载模型，use_lora = False


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


已生成：5/50
已生成：10/50
已生成：15/50
已生成：20/50
已生成：25/50
已生成：30/50
已生成：35/50
已生成：40/50
已生成：45/50
已生成：50/50

开始加载模型，use_lora = True
已生成：5/50
已生成：10/50
已生成：15/50
已生成：20/50
已生成：25/50
已生成：30/50
已生成：35/50
已生成：40/50
已生成：45/50
已生成：50/50


#### 计算 CER

In [6]:
# 下面计算的是批量总体 CER：
# 总错误字符数 ÷ 总参考答案字符数
base_cer = jiwer.cer(
    reference=references,
    hypothesis=base_predictions,
)

lora_cer = jiwer.cer(
    reference=references,
    hypothesis=lora_predictions,
)

print("\n" + "=" * 50)
print("原始 Whisper CER：", base_cer)
print("Smoke LoRA CER：  ", lora_cer)
print("CER 降低量：       ", base_cer - lora_cer)
print("=" * 50)
print("原始模型耗时：", round(base_time, 2), "秒")
print("LoRA 模型耗时：", round(lora_time, 2), "秒")


原始 Whisper CER： 0.1373873873873874
Smoke LoRA CER：   0.13626126126126126
CER 降低量：        0.0011261261261261424
原始模型耗时： 113.21 秒
LoRA 模型耗时： 129.62 秒


#### 生成每条样本的对比表

In [7]:
rows = []

for index, (
    reference,
    base_prediction,
    lora_prediction,
) in enumerate(
    zip(
        references,
        base_predictions,
        lora_predictions,
    ),
    start=1,
):
    # 每条样本单独计算 CER
    base_sample_cer = jiwer.cer(
        reference=reference,
        hypothesis=base_prediction,
    )

    lora_sample_cer = jiwer.cer(
        reference=reference,
        hypothesis=lora_prediction,
    )

    rows.append({
        "序号": index,
        "标准答案": reference,
        "原始模型输出": base_prediction,
        "LoRA输出": lora_prediction,
        "原始CER": round(base_sample_cer, 4),
        "LoRA CER": round(lora_sample_cer, 4),
        "LoRA改善": round(
            base_sample_cer - lora_sample_cer,
            4,
        ),
    })

results_df = pd.DataFrame(rows)

# 按 LoRA 改善量从高到低排序
results_df = results_df.sort_values(
    by="LoRA改善",
    ascending=False,
)

display(results_df)

,序号,标准答案,原始模型输出,LoRA输出,原始CER,LoRA CER,LoRA改善
16,17,该村庄建于多米尼克的陡峭地带上。,该村庄建于多米尼克的陡峭地带上,该村庄建于多米尼克的陡峭地带上。,0.0625,0.0000,0.0625
29,30,乌拉嘎龙的正模标本是一个部份齿骨。,乌拉卡龙的正膜标本是一个部分齿骨,乌拉卡龙的正膜标本是一个部分齿骨。,0.2353,0.1765,0.0588
0,1,生于今宁波余姚市南城学弄至东朝街莫家弄。,生于金宁波于遥是南城学弄至东朝皆莫加弄,生于金宁波于遥是南城学弄至东朝皆莫加弄。,0.3500,0.3000,0.0500
20,21,里克·瑞斯克拉是曾经参加越南战争的美军上校。,李克·瑞斯克拉是曾经参加越南战争的美军上校,李克·瑞斯克拉是曾经参加越南战争的美军上校。,0.0909,0.0455,0.0455
37,38,本站是北京地铁少数几个采用侧式叠式站台的车站之一。,本站是北京地铁少数几个采用测试迭式站台的车站之一,本站是北京地铁少数几个采用测试迭式站台的车站之一。,0.1600,0.1200,0.0400
45,46,符号表是一个在符号名称与它们的记忆体位置间的查询表格。,符号表是一个在符号名称与他们的记忆体位置间的查询表格。,符号表是一个在符号名称与他们的记忆体位置间的查询表格。,0.0370,0.0370,0.0000
44,45,战役以魏得胜告终。,战役已未得胜告终,战役已为得胜告终,0.3333,0.3333,0.0000
28,29,石灰拟平藓为平藓科拟平藓属下的一个种。,石灰拟平显为平显科拟平显属下的一个种。,石灰拟平显为平显科拟平显属下的一个种。,0.1579,0.1579,0.0000
48,49,本项目是首次成为伤残奥林匹克运动会比赛项目。,本项目是首次成为商残奥林匹克运动会比赛项目。,本项目是首次成为商残奥林匹克运动会比赛项目。,0.0455,0.0455,0.0000
30,31,黑色铁角蕨为铁角蕨科铁角蕨属下的一个变种。,"黑色铁角角为铁角角科,铁角角属下的一个变种。","黑色铁角角为铁角角科,铁角角属下的一个变种。",0.1905,0.1905,0.0000


#### 统计 LoRA 的胜平负
- LoRA CER < 原始 CER：微调有收益，可以继续训练
- LoRA CER ≈ 原始 CER：没有明显收益，不需要继续 3 epochs
- LoRA CER > 原始 CER：微调伤害模型，应停止使用该 adapter

重点不只看总体 CER，还要看 LoRA 更好的样本 / 更差的样本。只有总体 CER 下降，并且大多数样本没有明显变差，这个 adapter 才值得保留。

In [8]:
lora_better = (
    results_df["LoRA改善"] > 0
).sum()

equal_count = (
    results_df["LoRA改善"] == 0
).sum()

lora_worse = (
    results_df["LoRA改善"] < 0
).sum()

print("LoRA 更好的样本：", lora_better)
print("两者相同的样本：", equal_count)
print("LoRA 更差的样本：", lora_worse)

LoRA 更好的样本： 5
两者相同的样本： 42
LoRA 更差的样本： 3


## 第三轮 1000 steps 跑完测试结果

In [20]:
import csv
from pathlib import Path

import jiwer
import librosa
import torch

from peft import PeftModel
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    BitsAndBytesConfig,
)

MODEL_ID = "openai/whisper-large-v3"

ADAPTER_DIR = (
    "/sigan/ai-ml/models/whisper-large-v3-zh-safe"
)

CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

# 加载 Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

# 加载量化基座
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True
    ),
    device_map="auto",
)

# 把 LoRA adapter 挂回 Whisper
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)
model.eval()

# 固定 Whisper 使用中文转写
model.generation_config.forced_decoder_ids = (
    processor.get_decoder_prompt_ids(
        language="chinese",
        task="transcribe",
    )
)


def transcribe(audio_path):
    # 加载并转为 Whisper 要求的 16kHz
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # 8-bit 模型内部偏置是 FP16
    inputs["input_features"] = (
        inputs["input_features"].half()
    )

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=1,
        )

    return processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )[0]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### 测试10条

In [24]:
with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    rows = list(csv.DictReader(
        handle,
        delimiter="\t",
    ))[:10]

references = []
predictions = []

for index, sample in enumerate(rows, start=1):
    audio_path = (
        CV_ROOT
        / "clips"
        / sample["path"]
    )

    # reference = sample["sentence"]
    reference = normalize_chinese(sample["sentence"],remove_punctuation=True)
    # prediction = transcribe(audio_path)
    prediction = normalize_chinese(transcribe(audio_path),remove_punctuation=True)

    references.append(reference)
    # references.append(normalize_chinese(reference,remove_punctuation=True))
    predictions.append(prediction)
    # references.append(normalize_chinese(prediction,remove_punctuation=True))

    print(f"\n样本 {index}")
    print("标准：", reference)
    print("预测：", prediction)
    print(
        "CER：",
        jiwer.cer(
            reference=reference,
            hypothesis=prediction,
        ),
    )

overall_cer = jiwer.cer(
    reference=references,
    hypothesis=predictions,
)

print("\n10 条总体 CER：", overall_cer)


样本 1
标准： 生于今宁波余姚市南城学弄至东朝街莫家弄
预测： 生于金宁波于遥是南城学弄至东朝皆莫加弄
CER： 0.3157894736842105

样本 2
标准： 硝酸钍溶液与草酸溶液反应沉淀出草酸钍
预测： 硝酸土溶液与草酸溶液反应沉淀出草酸土
CER： 0.1111111111111111

样本 3
标准： 日本的全称为株式会社平凡社
预测： 日本的全程为都市会社平反社
CER： 0.3076923076923077

样本 4
标准： 亿的单位与上下数可参考下表
预测： e的单位与上下数和参考下表
CER： 0.15384615384615385

样本 5
标准： 带广志的名义绘画成人向漫画
预测： 戴广志的名义绘画成人像漫画
CER： 0.15384615384615385

样本 6
标准： 满洲国陆军军官学校位于新京
预测： 满洲国陆军军官学校位于新京
CER： 0.0

样本 7
标准： 埃皮奈人口变化图示
预测： ip内人口变化图示
CER： 0.3333333333333333

样本 8
标准： 相传民国十一年桥的部分木结构桥身曾被洪水冲走
预测： 相传民国十一年桥的部分木结构桥身曾被洪水冲走
CER： 0.0

样本 9
标准： 根城日本之古城
预测： 根城日本之古城
CER： 0.0

样本 10
标准： 物理学者做实验尚未发现任何违背玻恩定则的量子行为
预测： 物理学者做实验尚未发现任何违背波恩定则的量子行为
CER： 0.041666666666666664

10 条总体 CER： 0.13245033112582782


### 结果判断

在“同一批通用中文朗读数据”上盲目增加训练步数，意义确实不大

## 第四轮 1000 steps 跑完测试结果

In [13]:
import csv
from pathlib import Path

import jiwer
import librosa
import torch

from peft import PeftModel
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    BitsAndBytesConfig,
)

MODEL_ID = "openai/whisper-large-v3"

ADAPTER_DIR = (
    "/sigan/ai-ml/models/whisper-large-v3-zh-safe-evalfix"
)

CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

# 加载 Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

# 加载量化基座
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True
    ),
    device_map="auto",
)

# 把 LoRA adapter 挂回 Whisper
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)
model.eval()

# 固定 Whisper 使用中文转写
model.generation_config.forced_decoder_ids = (
    processor.get_decoder_prompt_ids(
        language="chinese",
        task="transcribe",
    )
)


def transcribe(audio_path):
    # 加载并转为 Whisper 要求的 16kHz
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # 8-bit 模型内部偏置是 FP16
    inputs["input_features"] = (
        inputs["input_features"].half()
    )

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=1,
        )

    return processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )[0]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### 测试10条

In [14]:
with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    rows = list(csv.DictReader(
        handle,
        delimiter="\t",
    ))[:10]

references = []
predictions = []

for index, sample in enumerate(rows, start=1):
    audio_path = (
        CV_ROOT
        / "clips"
        / sample["path"]
    )

    # reference = sample["sentence"]
    reference = normalize_chinese(sample["sentence"],remove_punctuation=True)
    # prediction = transcribe(audio_path)
    prediction = normalize_chinese(transcribe(audio_path),remove_punctuation=True)

    references.append(reference)
    # references.append(normalize_chinese(reference,remove_punctuation=True))
    predictions.append(prediction)
    # references.append(normalize_chinese(prediction,remove_punctuation=True))

    print(f"\n样本 {index}")
    print("标准：", reference)
    print("预测：", prediction)
    print(
        "CER：",
        jiwer.cer(
            reference=reference,
            hypothesis=prediction,
        ),
    )

overall_cer = jiwer.cer(
    reference=references,
    hypothesis=predictions,
)

print("\n10 条总体 CER：", overall_cer)


样本 1
标准： 生于今宁波余姚市南城学弄至东朝街莫家弄
预测： 生于金宁波于遥是南城学弄至东朝皆莫加弄
CER： 0.3157894736842105

样本 2
标准： 硝酸钍溶液与草酸溶液反应沉淀出草酸钍
预测： 硝酸土溶液与草酸溶液反应沉淀出草酸土
CER： 0.1111111111111111

样本 3
标准： 日本的全称为株式会社平凡社
预测： 日本的全称为都市会社平反社
CER： 0.23076923076923078

样本 4
标准： 亿的单位与上下数可参考下表
预测： e的单位与上下数和参考下表
CER： 0.15384615384615385

样本 5
标准： 带广志的名义绘画成人向漫画
预测： 戴广志的名义绘画成人像漫画
CER： 0.15384615384615385

样本 6
标准： 满洲国陆军军官学校位于新京
预测： 满洲国陆军军官学校位于新京
CER： 0.0

样本 7
标准： 埃皮奈人口变化图示
预测： ip内人口变化图示
CER： 0.3333333333333333

样本 8
标准： 相传民国十一年桥的部分木结构桥身曾被洪水冲走
预测： 相传民国十一年桥的部分木结构桥身曾被洪水冲走
CER： 0.0

样本 9
标准： 根城日本之古城
预测： 根城日本之古城
CER： 0.0

样本 10
标准： 物理学者做实验尚未发现任何违背玻恩定则的量子行为
预测： 物理学者做实验尚未发现任何违背波恩定则的量子行为
CER： 0.041666666666666664

10 条总体 CER： 0.12582781456953643


### 结果判断